# Modul 5: Detekcja AprilTag — precyzyjna lokalizacja obiektow

W tym notebooku zaimplementujemy detekcje **markerow fiducjalnych AprilTag** — alternatywe dla YOLO,
ktora daje **precyzyjna 6DOF poze** (pozycja + orientacja) kazdego markera.

### Zastosowania
- **Logistyka / warehouse** — identyfikacja kartonow po unikalnym ID
- **Pick-and-place** — precyzyjna poza do chwytania (robot G1 + Dex3)
- **Kalibracja hand-eye** — znana geometria markera
- **Dokowanie robota** — precyzyjne pozycjonowanie wzgledem stacji

In [ ]:
!pip install dt-apriltags opencv-python-headless numpy matplotlib -q

## AprilTag vs YOLO — kiedy co uzywac

| Cecha | AprilTag / ArUco | YOLO |
|-------|-----------------|------|
| **Wynik** | 6DOF poza (pozycja + rotacja) | 2D bounding box + klasa |
| **Latencja** | ~5 ms (CPU) | 10-20 ms (GPU / TensorRT) |
| **Trening** | Nie wymaga | Wymaga etykietowanych danych |
| **Unikalne ID** | Tak (zakodowane w markerze) | Nie (tylko klasa) |
| **Deterministycznosc** | 100% powtarzalnosc | Probabilistyczny |
| **Ograniczenia** | Wymaga naklejenia markera | Dowolny obiekt |

**Regula kciuka:** Jesli kontrolujesz srodowisko i potrzebujesz precyzji — uzyj markerow.
Jesli obiekty sa nieznane — uzyj YOLO.

In [ ]:
# Porownanie w formie tabeli
comparison = [
    ["Wynik",             "6DOF poza (xyz + rotacja)",  "2D bbox + klasa"],
    ["Latencja",          "~5 ms (CPU)",                "10-20 ms (GPU)"],
    ["Trening",           "Brak",                       "50-100+ zdjec"],
    ["Unikalne ID",       "Tak (w markerze)",           "Nie"],
    ["Deterministycznosc","100%",                       "Probabilistyczny"],
    ["Ograniczenie",      "Wymaga naklejenia markera",  "Dowolny obiekt"],
]

print(f"{'Cecha':<22} {'AprilTag/ArUco':<30} {'YOLO'}")
print("-" * 75)
for row in comparison:
    print(f"  {row[0]:<20} {row[1]:<30} {row[2]}")

## Generowanie testowego obrazu z tagami

Poniewaz w Colab nie mamy fizycznej kamery, wygenerujemy syntetyczny obraz z 3 namalowanymi
kwadratami imitujacymi markery AprilTag. Detektor powinien je rozpoznac.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def create_synthetic_tags(width=640, height=480):
    """Generuj obraz z 3 syntetycznymi tagami (czarne kwadraty z bialym border)."""
    img = np.full((height, width, 3), 200, dtype=np.uint8)
    positions = [(150, 150), (350, 200), (500, 300)]
    sizes = [60, 50, 45]
    for (px, py), sz in zip(positions, sizes):
        half = sz // 2
        # Biale obramowanie
        cv2.rectangle(img, (px - half - 6, py - half - 6),
                       (px + half + 6, py + half + 6), (255, 255, 255), -1)
        # Czarny kwadrat (marker)
        cv2.rectangle(img, (px - half, py - half),
                       (px + half, py + half), (0, 0, 0), -1)
        # Wewnetrzny wzor — siatka
        step = sz // 4
        for i in range(1, 4):
            for j in range(1, 4):
                if (i + j) % 2 == 0:
                    bx = px - half + i * step
                    by = py - half + j * step
                    cv2.rectangle(img, (bx, by), (bx + step, by + step),
                                  (255, 255, 255), -1)
    return img

# Alternatywa: uzyj OpenCV ArUco do generowania prawdziwych markerow
def create_aruco_image(width=640, height=480):
    """Generuj obraz z 3 prawdziwymi markerami ArUco (cv2.aruco)."""
    img = np.full((height, width, 3), 200, dtype=np.uint8)
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_APRILTAG_36h11)
    positions = [(120, 120), (320, 170), (470, 270)]
    sizes = [80, 70, 60]
    for idx, ((px, py), sz) in enumerate(zip(positions, sizes)):
        marker = cv2.aruco.generateImageMarker(aruco_dict, idx, sz)
        marker_bgr = cv2.cvtColor(marker, cv2.COLOR_GRAY2BGR)
        # Umieszczenie markera w obrazie
        y1, y2 = py, py + sz
        x1, x2 = px, px + sz
        if y2 <= height and x2 <= width:
            img[y1:y2, x1:x2] = marker_bgr
    return img

# Probuj ArUco (prawdziwe markery), fallback na syntetyczne
try:
    test_img = create_aruco_image()
    method = "ArUco (DICT_APRILTAG_36h11)"
except Exception:
    test_img = create_synthetic_tags()
    method = "syntetyczne kwadraty"

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
plt.title(f"Testowy obraz z tagami ({method})")
plt.axis('off')
plt.show()
print(f"Rozmiar obrazu: {test_img.shape[1]}x{test_img.shape[0]}")
print(f"Metoda generowania: {method}")

## Detekcja i 6DOF estymacja pozy

Uruchamiamy detektor AprilTag z estymacja pozy.
Potrzebujemy parametrow intrinsic kamery (fx, fy, cx, cy) oraz rozmiar markera w metrach.

In [ ]:
from dt_apriltags import Detector

# Parametry kamery RealSense D435i (typowe wartosci)
camera_params = [615.0, 615.0, 320.0, 240.0]  # fx, fy, cx, cy
tag_size = 0.08  # 8 cm

detector = Detector(
    families="tag36h11",
    nthreads=4,
    quad_decimate=1.0,
)

# Konwersja do grayscale i detekcja
gray = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)
results = detector.detect(
    gray,
    estimate_tag_pose=True,
    camera_params=camera_params,
    tag_size=tag_size,
)

print(f"Wykryto {len(results)} tagow AprilTag\n")

# Wyswietl wyniki
for r in results:
    t = r.pose_t.flatten() if r.pose_t is not None else [0, 0, 0]
    print(f"  Tag ID={r.tag_id:>3d}")
    print(f"    Srodek (px):  ({r.center[0]:.1f}, {r.center[1]:.1f})")
    print(f"    Pozycja 3D:   x={t[0]:+.4f}m  y={t[1]:+.4f}m  z={t[2]:+.4f}m")
    print(f"    Odleglosc:    {t[2]:.3f} m")
    if r.pose_R is not None:
        print(f"    Rotacja (R):  {r.pose_R[0].round(3)}")
    print()

# Rysuj detekcje na obrazie
vis = test_img.copy()
for r in results:
    pts = r.corners.astype(int)
    for i in range(4):
        cv2.line(vis, tuple(pts[i]), tuple(pts[(i + 1) % 4]), (0, 255, 0), 2)
    cx_t, cy_t = int(r.center[0]), int(r.center[1])
    cv2.circle(vis, (cx_t, cy_t), 5, (0, 0, 255), -1)
    cv2.putText(vis, f"ID:{r.tag_id}", (cx_t + 8, cy_t - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    if r.pose_t is not None:
        dist = r.pose_t.flatten()[2]
        cv2.putText(vis, f"d={dist:.2f}m", (cx_t + 8, cy_t + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 200, 0), 1)

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"Detekcja AprilTag — {len(results)} tag(ow)")
plt.axis('off')
plt.tight_layout()
plt.show()

## Transformacja do ukladu robota

Detektor zwraca poze w **ukladzie kamery** (camera_link). Zeby robot mogl uzyc tych danych
do nawigacji lub chwytania, musimy przetransformowac pozycje do **ukladu robota** (base_link).

Na G1 kamera D435i jest zamontowana na glowie:
- ~5 cm do przodu od base_link
- ~1.2 m do gory
- ~10 stopni tilt w dol (pitch = 0.1745 rad)

Transformacja to mnozenie macierzy 4x4:
```
T_base_object = T_base_cam @ T_cam_object
```

In [ ]:
def make_T_base_cam(tx=0.05, ty=0.0, tz=1.2, pitch=0.1745):
    """Macierz transformacji camera_link -> base_link.
    
    Parametry montazu kamery na G1:
      tx=0.05m do przodu, tz=1.2m do gory, pitch=10deg tilt w dol.
    """
    cp, sp = np.cos(pitch), np.sin(pitch)
    T = np.array([
        [1.0, 0.0,  0.0, tx],
        [0.0,  cp, -sp, ty],
        [0.0,  sp,  cp, tz],
        [0.0, 0.0,  0.0, 1.0],
    ])
    return T


def camera_to_base_link(pose_t, pose_R, T_base_cam):
    """Przelicz poze z ukladu kamery na uklad base_link."""
    T_cam_obj = np.eye(4)
    T_cam_obj[:3, :3] = pose_R
    T_cam_obj[:3, 3] = pose_t.flatten()
    T_base_obj = T_base_cam @ T_cam_obj
    return T_base_obj[:3, 3], T_base_obj[:3, :3]


# Transformacja wynikow
T_base_cam = make_T_base_cam()

print("Macierz T_base_cam (camera_link -> base_link):")
print(T_base_cam.round(4))
print()

print("Pozycje tagow w ukladzie base_link:")
print("=" * 60)
for r in results:
    if r.pose_t is not None and r.pose_R is not None:
        pos_cam = r.pose_t.flatten()
        pos_base, _ = camera_to_base_link(r.pose_t, r.pose_R, T_base_cam)
        print(f"  Tag ID={r.tag_id}:")
        print(f"    camera_link:  x={pos_cam[0]:+.3f}  y={pos_cam[1]:+.3f}  z={pos_cam[2]:+.3f} m")
        print(f"    base_link:    x={pos_base[0]:+.3f}  y={pos_base[1]:+.3f}  z={pos_base[2]:+.3f} m")
        print()

# Na prawdziwym robocie uzyjesz tf2_ros:
# ros2 run tf2_ros static_transform_publisher \
#   --x 0.05 --y 0.0 --z 1.2 \
#   --roll 0.0 --pitch 0.1745 --yaw 0.0 \
#   --frame-id base_link --child-frame-id camera_link

## Pipeline logistyczny

AprilTag laczy sie z innymi modulami kursu w kompletny pipeline logistyczny:

1. **Modul 5 (Percepcja)** — detekcja AprilTag na kartonie → 6DOF poza
2. **Modul 6 (Nawigacja)** — Nav2 planuje sciezke do kartonu
3. **Modul 4 (Dex3)** — chwytanie kartonu z precyzyjna poza z markera

```
Kamera RealSense
      |
      v
AprilTag detect → ID + 6DOF poza
      |
      v
TF transform → pozycja w ukladzie mapy
      |
      v
Nav2 navigate_to_pose → robot podchodzi do kartonu
      |
      v
Dex3 grasp (precyzyjne chwytanie z orientacja z markera)
```

### Przewaga markerow w logistyce
- **Unikalne ID** — kazdy karton ma inny numer, mozna sledzic konkretne paczki
- **Precyzja chwytania** — 6DOF poza (nie tylko bbox) pozwala na chwyt pod wlasciwym katem
- **Zero false positives** — detekcja jest deterministyczna, brak pomylek

## Nastepne kroki

1. **Wydrukuj prawdziwe tagi** — [generator tagow tag36h11](https://github.com/AprilRobotics/apriltag-imgs/tree/master/tag36h11) — wydrukuj ID 0-5, naklej na kartony
2. **Testuj na prawdziwej kamerze** — podlacz RealSense D435i i uruchom `ex5_apriltag_detect.py --camera 0`
3. **Integracja z Nav2** — opublikuj poze tagu jako goal_pose dla Nav2 (Modul 6)
4. **Kalibracja hand-eye** — uzyj znanej geometrii markera do kalibracji kamery wzgledem nadgarstka (Modul 4)
5. **Hybrid pipeline** — polacz YOLO (wykrycie kartonu) + AprilTag (precyzyjna poza do chwytania)

### Przydatne linki
- [dt-apriltags PyPI](https://pypi.org/project/dt-apriltags/)
- [AprilTag paper (Olson 2011)](https://april.eecs.umich.edu/papers/details.php?name=olson2011tags)
- [OpenCV ArUco tutorial](https://docs.opencv.org/4.x/d5/dae/tutorial_aruco_detection.html)